# sales_intelligence.ipynb

### Sales KPIs
- Total Sales
- Sales Growth
- Average Basket
- Performance:
B2B vs B2C
Payment Methods
Delivery Methods
Sales Status
- discount impact


This notebook analyses the sales of Energical, and gives all the necessery KPIs and analysis to have clean insights and make business decisions.

In [626]:
import pandas as pd

import plotly
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from plotly.subplots import make_subplots

import numpy as np

In [627]:
clean_transactions=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_transactions.csv")
clean_orders=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_orders.csv")
clean_customers=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_customers.csv")
clean_catalogue=pd.read_csv(r"C:\Users\PCPRODZ\Desktop\energical-decision-platform\data\CleanData\clean_catalogue.csv")


In [628]:
clean_customers.rename(columns={
    "Code Client": "customer_id_stage"
    }, inplace=True)

In [629]:
valid_sales=clean_orders[clean_orders["order_status"].isin(["Terminée", "Partiellement remboursée"])]

## Sales number

In [630]:
def sale_number(valid_sales):
    sales_number=valid_sales['order_total'].sum()
    return sales_number

## Sales growth/trend

In [631]:
def Sales_growth(current_sales_number,previous_sales_number):
    if previous_sales_number==0:
        return None
    else:
        sales_growth=((current_sales_number-previous_sales_number)/previous_sales_number)*100
    return sales_growth

## Average basket value

In [632]:
def avg_basket_value(clean_customers):
    clean_customers["average_basket"] = (
        clean_customers["total_amount"] / clean_customers["orders_count"]
    )

    return clean_customers

## Sales performance

In [633]:
#performance per method of payment
def performance_per_mop(Valid_sales):
    performance = (
        Valid_sales
        .groupby("payment_method_group")
        .agg(
            Revenue=("order_total_amount", "sum"),
            Orders=("order_id_stage", "count"),
            Average_Basket=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return performance

In [634]:
shipping_per_order = (
    clean_transactions[["order_id_stage", "shipping_method"]]
    .drop_duplicates()
)
orders_shipping=(
    valid_sales
    .merge(shipping_per_order, on="order_id_stage", how="left")
)

In [635]:
#performance per delivery method
def performance_per_delivery_method(orders_shipping):
    performance = (
        orders_shipping.groupby("shipping_method").agg(
            Revenue=("order_total_amount", "sum"),
            Orders=("order_id_stage", "count"),
            Average_Basket=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return performance


In [636]:
#performance per customer type
def performance_per_customer_type(clean_customers):
    performance=(
        clean_customers.groupby("customer_type_inferred").agg(
            Revenue=("total_amount", "sum"),
            Orders=("orders_count", "count"),
            Average_Basket=("average_basket", "mean")
        )
        .reset_index()
    )
    return performance

## Return rate vs completed sales

In [637]:
status_mapping = {
    "Terminée": "Completed",
    "En cours": "In Progress",
    "En livraison par NOEST": "In Progress",

    "En attente": "Pending",
    "Attente paiement": "Pending",
    "Partiellement payé": "Pending",

    "Retour NOEST": "Returned",
    "Remboursée": "Refunded",
    "Partiellement remboursée": "Partially Refunded"
}
clean_orders["sales_status"] = clean_orders["order_status"].map(status_mapping)

In [638]:
def sales_status(clean_orders):

    sales_status = (
        clean_orders.groupby("sales_status")
        .agg(
            orders_count=("order_id_stage", "count")
        )
        .reset_index()
    )

    sales_status["percentage"] = (
        sales_status["orders_count"]
        / sales_status["orders_count"].sum()
    ) * 100

    sales_status["percentage"] = sales_status["percentage"].round(2)

    return sales_status

## discount impact

In [639]:
clean_transactions.rename(columns={
    "has_negative_price": "has_discount"
}, inplace=True)

In [640]:
clean_transactions[clean_transactions["has_discount"]==True]

,order_id_stage,customer_id_stage,order_date,wilaya_raw,wilaya_normalized,geo_quality_flag,customer_type_inferred,sku,product_name,sku_quality,...,line_total,order_status,order_total_amount,payment_method,sales_channel,shipping_method,total_weight,shipping_cost,has_discount,free_shipping
4047,CMD_S001942,CLT_S001456,2023-12-16,Béjaïa,Béjaïa,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4655.00,Terminée,69000.0,Cash on Delivery,web,Home Delivery,4.3485,0.0,True,True
4198,CMD_S002001,CLT_S001497,2023-12-25,Timimoun,Timimoun,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4655.00,Terminée,51680.0,Cash on Delivery,web,Home Delivery,5.4485,0.0,True,True
4388,CMD_S002097,CLT_S001576,2024-01-12,Constantine,Constantine,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4585.17,Terminée,43340.0,CIB & Edahabia Card,web,Home Delivery,3.5485,0.0,True,True
4469,CMD_S002137,CLT_S001604,2024-01-19,Tébessa,Tébessa,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4655.00,Terminée,44000.0,Cash on Delivery,web,Home Delivery,3.5485,0.0,True,True
4515,CMD_S002156,CLT_S001616,2024-01-22,Ouargla,Ouargla,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4655.00,Terminée,44000.0,Cash on Delivery,web,Home Delivery,3.5485,0.0,True,True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18746,CMD_S008578,CLT_S000279,2026-05-11,Oran,Oran,ok,B2C,SKU-38086,Pack Ruban LED RGB 1 ligne 8W (Multi-couleurs)...,auto_generated,...,-20.00,Terminée,2330.0,Cash on Delivery,web,Pickup Point,0.0000,350.0,True,False
18813,CMD_S008606,CLT_S005402,2026-05-13,El Oued,El Oued,ok,B2C,VFE01C-VFE04-2,Pack Visiophone Complet 472-C,not_in_catalog,...,-4725.00,Terminée,43300.0,Cash on Delivery,web,Pickup Point,3.5485,0.0,True,True
18884,CMD_S008637,CLT_S005418,2026-05-17,Alger,Alger,ok,B2C,SKU-38086,Pack Ruban LED RGB 1 ligne 8W (Multi-couleurs)...,auto_generated,...,-20.00,Retour NOEST,1720.0,Cash on Delivery,web,Pickup Point,0.0000,250.0,True,False
19144,CMD_S008736,CLT_S005473,2026-06-01,Saïda,Saïda,ok,B2C,SKU-38086,Pack Ruban LED RGB 1 ligne 8W (Multi-couleurs)...,auto_generated,...,-19.60,Terminée,2016.0,CIB & Edahabia Card,web,Pickup Point,0.0000,350.0,True,False


In [641]:
def discount_impact(clean_transactions, clean_orders):

    # Create one row per order indicating whether it contains a discount
    discount_orders = (
        clean_transactions.groupby("order_id_stage")["has_discount"]
        .any()
        .reset_index()
        .rename(columns={"has_discount": "order_has_discount"})
    )

    # Merge the flag into the orders table
    orders = clean_orders.merge(
        discount_orders,
        on="order_id_stage",
        how="left"
    )

    # Orders with no discount line become False
    orders["order_has_discount"] = orders["order_has_discount"].fillna(False)

    # Aggregate at the order level
    impact = (
        orders.groupby("order_has_discount")
        .agg(
            Total_Revenue=("order_total_amount", "sum"),
            Total_Orders=("order_id_stage", "count"),
            Average_Order_Value=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return impact

In [642]:
discount_impact(clean_transactions, clean_orders)

C:\Users\PCPRODZ\AppData\Local\Temp\ipykernel_18460\449299466.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orders["order_has_discount"] = orders["order_has_discount"].fillna(False)


,order_has_discount,Total_Revenue,Total_Orders,Average_Order_Value
0,False,3.238040e+08,9184,35257.408637
1,True,2.282700e+06,61,37421.304918


## Free delivery impact

In [643]:
def free_del_impact(clean_transactions, clean_orders):
    free_shipping_orders =(
        clean_transactions.groupby("order_id_stage")["free_shipping"]
                .any()
                .reset_index()
                .rename(columns={"free_shipping": "order_has_free_shipping"})
    )
    
    orders = clean_orders.merge(
        free_shipping_orders,
        on="order_id_stage",
        how="left"
    )
    

    orders["order_has_free_shipping"] = orders["order_has_free_shipping"].fillna(False)
    impact = (
        orders.groupby("order_has_free_shipping")
        .agg(
            Total_Revenue=("order_total_amount", "sum"),
            Total_Orders=("order_id_stage", "count"),
            Average_Order_Value=("order_total_amount", "mean")
        )
        .reset_index()
    )

    return impact

In [644]:
free_del_impact(clean_transactions, clean_orders)

C:\Users\PCPRODZ\AppData\Local\Temp\ipykernel_18460\1915653248.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orders["order_has_free_shipping"] = orders["order_has_free_shipping"].fillna(False)


,order_has_free_shipping,Total_Revenue,Total_Orders,Average_Order_Value
0,False,1.123292e+08,7438,15102.073404
1,True,2.137575e+08,1807,118294.144184


# Graphs

### sales number

In [645]:
valid_sales["order_date"]=valid_sales["order_date"].astype("datetime64[ns]")

C:\Users\PCPRODZ\AppData\Local\Temp\ipykernel_18460\2935984948.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  valid_sales["order_date"]=valid_sales["order_date"].astype("datetime64[ns]")


In [646]:
sales = (
    valid_sales.groupby(
        valid_sales["order_date"].dt.to_period("M")
    )["order_total_amount"]
    .sum()
    .reset_index()
)
sales["order_date"] = sales["order_date"].astype(str)
fig = px.line(
    sales,
    x="order_date",
    y="order_total_amount",
    markers='markers',
    title="Sales Revenue"
)

fig.show()

## B2B VS B2C graph

In [647]:
b2b_b2c = (
    clean_customers.groupby("customer_type_inferred")
    .agg(
        Revenue=("total_amount", "sum"),
        Customers=("customer_id_stage", "count"),
        Average_Basket=("average_basket", "mean")
    )
    .reset_index()
)

In [648]:
perf = performance_per_customer_type(clean_customers)

fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["customer_type_inferred"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [649]:
perf = performance_per_mop(valid_sales)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["payment_method_group"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [650]:
perf = performance_per_delivery_method(orders_shipping)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=perf["shipping_method"], y=perf["Average_Basket"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="B2B vs B2C Performance", showlegend=False)
fig.show()

In [651]:
sales_status(clean_orders)

,sales_status,orders_count,percentage
0,Completed,8807,95.26
1,In Progress,251,2.71
2,Partially Refunded,24,0.26
3,Pending,11,0.12
4,Refunded,21,0.23
5,Returned,131,1.42


In [652]:
fig= px.pie(
    sales_status(clean_orders),
    names='sales_status',
    values='orders_count',
    
)
fig.show()

In [653]:
impact = free_del_impact(clean_transactions, clean_orders)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Total_Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Total_Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=impact["order_has_free_shipping"], y=impact["Average_Order_Value"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="Free Shipping Impact", showlegend=False)
fig.show()

C:\Users\PCPRODZ\AppData\Local\Temp\ipykernel_18460\1915653248.py:16: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orders["order_has_free_shipping"] = orders["order_has_free_shipping"].fillna(False)


In [654]:
impact = discount_impact(clean_transactions, clean_orders)
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=("Revenue", "Orders", "Average Basket")
)

fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Total_Revenue"], name="Revenue"),
    row=1, col=1
)
fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Total_Orders"], name="Orders"),
    row=1, col=2
)
fig.add_trace(
    go.Bar(x=impact["order_has_discount"], y=impact["Average_Order_Value"], name="Average Basket"),
    row=1, col=3
)

fig.update_layout(title_text="Discount Impact", showlegend=False)
fig.show()

C:\Users\PCPRODZ\AppData\Local\Temp\ipykernel_18460\449299466.py:19: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  orders["order_has_discount"] = orders["order_has_discount"].fillna(False)
